# Extend frozen 00c resources with pooled LCC localization

This notebook copies and verifies the completed 00c resource bundle, then runs the frozen 00-v2 SCRFD LCC policy on **training** and **development** only. The existing official evaluation cache and manifest are reused byte for byte. No PAD model training or evaluation occurs here.

Set the four paths in CONFIG. Use `SPLIT_FILTER=None` for both splits, or `['training']` / `['development']` to finish one split in a resumable output directory. The final pooled artifacts and ZIP are produced once both split caches are complete. Raw LCC images and the exact SCRFD model must be mounted in Kaggle.


In [ ]:
# CONFIG: mount the completed 00c bundle, common LCC root, and exact SCRFD-500M model.
BASE_RESOURCE_DIR='/kaggle/input/.../minifasnet_binary_crossdomain_resources_v2'
LCC_ROOT='/kaggle/input/.../LCC_FASD'
SCRFD_MODEL_PATH='/kaggle/input/.../det_500m.onnx'
OUTPUT_ROOT='/kaggle/working/minifasnet_binary_crossdomain_resources_v2_lcc_pooled'
SPLIT_FILTER=None # None: training + development; or ['training'], ['development'].
INSTALL_DETECTOR_DEPS=True # Install the frozen InsightFace major version and a CUDA-compatible ONNX Runtime.
ONNXRUNTIME_GPU_SPEC='onnxruntime-gpu==1.26.0' # CUDA 12.8 + cuDNN 9; Kaggle PyTorch 2.11.0+cu128.
CACHE_SAVE_EVERY=200
PARENT_CONTRACT_SHA='aa67032a79627636d967dc885202540a34410c2b4d64522a5604d3aedff85b2d'
V2_CONTRACT_SHA='2f1f6eb6351550d491e4fb59e2151b1ccd9b000c0985cd32f12621f9d58c5f91'
DETECTOR_SHA='5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a'
LOCALIZATION_NOTEBOOK_SHA='e4c8565f48be3ee82228b5e88666c3cb0001ce92295835e973253fc3087b495d'
EXTENSION_PROTOCOL_SHA='d809c72cbdc0fafeebb718e07a4b7cea745ba132fc2b4c2905c4e2239704ecd2'



In [ ]:
import hashlib, json, os, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path

required_00c=(
 'config.json','provenance.json','manifest_fingerprints.json','PAD_DATA_AND_EVALUATION_CONTRACT_v2_CROSSDOMAIN_BINARY.md',
 'bbox/celeba_bbox_cache_full.json','bbox/lcc_official_evaluation_bbox_cache.json','labels/celeba_auxiliary_labels.csv',
 'manifests/mini_train10k.csv','manifests/mini_val3k.csv','manifests/crossdomain_train_full.csv',
 'manifests/crossdomain_celeba_source_dev_full.csv','manifests/lcc_official_evaluation_full.csv')
missing_00c=[rel for rel in required_00c if not (Path(BASE_RESOURCE_DIR)/rel).is_file()]
if missing_00c:
    raise FileNotFoundError('BASE_RESOURCE_DIR must point to the completed 00c binary-crossdomain bundle; missing: '+', '.join(missing_00c))

if SPLIT_FILTER is not None and (not isinstance(SPLIT_FILTER,(list,tuple)) or not SPLIT_FILTER or
    len(set(SPLIT_FILTER))!=len(SPLIT_FILTER) or set(SPLIT_FILTER)-{'training','development'}):
    raise ValueError("SPLIT_FILTER must be None or unique values from training/development")
if not 100<=CACHE_SAVE_EVERY<=250:raise ValueError('CACHE_SAVE_EVERY must be 100..250')
import torch
print('PyTorch:',torch.__version__,'CUDA build:',torch.version.cuda,
      'CUDA available:',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is unavailable. Set Notebook Settings > Accelerator > GPU, then restart the session.')
from importlib.metadata import PackageNotFoundError, version as package_version
def installed_version(name):
    try:return package_version(name)
    except PackageNotFoundError:return None
if INSTALL_DETECTOR_DEPS:
    ready=(installed_version('insightface')=='2.0' and
           installed_version('onnxruntime-gpu')=='1.26.0' and
           installed_version('onnxruntime') is None)
    if ready:
        print('Reusing InsightFace 2.0 and ONNX Runtime GPU 1.26.0 already installed in this session')
    else:
        if 'onnxruntime' in sys.modules or 'insightface' in sys.modules:
            raise RuntimeError('ONNX Runtime/InsightFace was already imported. Restart the Kaggle session before reinstalling.')
        # InsightFace declares the CPU onnxruntime dependency. Replace it AFTER InsightFace installation.
        subprocess.check_call([sys.executable,'-m','pip','install','-q','insightface==2.0'])
        subprocess.check_call([sys.executable,'-m','pip','uninstall','-y','onnxruntime','onnxruntime-gpu'])
        subprocess.check_call([sys.executable,'-m','pip','install','-q',ONNXRUNTIME_GPU_SPEC])

import cv2, numpy as np, pandas as pd
import insightface, onnxruntime as ort
try:
    cpu_ort_version=package_version('onnxruntime')
except PackageNotFoundError:
    cpu_ort_version=None
if cpu_ort_version is not None:
    raise RuntimeError('CPU onnxruntime is still installed ('+cpu_ort_version+'); restart and rerun the installation cell.')
gpu_ort_version=package_version('onnxruntime-gpu')
if gpu_ort_version!=ort.__version__:
    raise RuntimeError('Imported ONNX Runtime differs from installed GPU package; restart the Kaggle session')
print('ONNX Runtime GPU:',ort.__version__,'providers:',ort.get_available_providers())
if hasattr(ort,'preload_dlls'):
    ort.preload_dlls() # Load the CUDA/cuDNN libraries supplied with the Kaggle PyTorch environment.
from insightface import model_zoo
from IPython.display import FileLink, display

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1048576),b''):h.update(block)
    return h.hexdigest()

def ordered_sha(keys):return hashlib.sha256('\n'.join(map(str,keys)).encode()).hexdigest()
def descriptor_sha(rows):return hashlib.sha256('\n'.join(f'{p}\t{y}' for p,y in rows).encode()).hexdigest()
def read_json(path):return json.loads(Path(path).read_text(),parse_constant=lambda s:(_ for _ in ()).throw(ValueError(s)))
def atomic_json_write(path,payload):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    fd,tmp=tempfile.mkstemp(prefix=path.name+'.',suffix='.tmp',dir=path.parent)
    try:
        with os.fdopen(fd,'w') as f:
            json.dump(payload,f,sort_keys=True,allow_nan=False,separators=(',',':'))
            f.write('\n');f.flush();os.fsync(f.fileno())
        os.replace(tmp,path)
    finally:
        if os.path.exists(tmp):os.unlink(tmp)

def hash_tree(folder):
    files={}
    for path in sorted(Path(folder).rglob('*')):
        if path.is_symlink():raise RuntimeError('Symlink in frozen base bundle: '+str(path))
        if path.is_file():files[path.relative_to(folder).as_posix()]=sha(path)
    if not files:raise RuntimeError('Empty base bundle')
    return files

def valid_scrfd_record(record,key=None):
    if record.get('status')!='VALID' or record.get('usable_for_pad') is not True or record.get('bbox_origin')!='SCRFD':return False
    if not str(record.get('bbox_source','')).startswith('SCRFD'):return False
    if key is not None and record.get('source_key')!=key:return False
    box=record.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in record:return False
    x1,y1,x2,y2=map(float,box)
    return bool(np.isfinite((x1,y1,x2,y2)).all() and x2>x1 and y2>y1)

def sanitized_invalid(status,key,reason):
    return {'status':status,'source_key':key,'bbox_origin':'NONE','usable_for_pad':False,
            'bbox_xyxy':None,'reason':reason}

def check_fresh_record(rec,key):
    if rec.get('status') not in {'VALID','DET_FAIL','SUSPICIOUS','UNREADABLE','INVALID_BBOX'}:
        raise RuntimeError('Unknown localization status: '+key)
    if rec.get('status')=='VALID':
        if not valid_scrfd_record(rec,key):raise RuntimeError('Unproven SCRFD bbox: '+key)
    elif rec.get('bbox_origin')!='NONE' or rec.get('usable_for_pad') is not False or rec.get('bbox_xyxy') is not None or rec.get('source_key')!=key:
        raise RuntimeError('Non-SCRFD fallback or invalid record: '+key)
BASE=Path(BASE_RESOURCE_DIR);ROOT=Path(OUTPUT_ROOT);LCC=Path(LCC_ROOT);MODEL=Path(SCRFD_MODEL_PATH)
if not BASE.is_dir() or not LCC.is_dir() or not MODEL.is_file():raise FileNotFoundError('Attach completed 00c bundle, common LCC root, and SCRFD model')
if ROOT.resolve()==BASE.resolve() or BASE.resolve() in ROOT.resolve().parents or ROOT.resolve() in BASE.resolve().parents:
    raise RuntimeError('OUTPUT_ROOT must be separate from BASE_RESOURCE_DIR')
for rel in required_00c:
    if not (BASE/rel).is_file():raise FileNotFoundError(BASE/rel)
base_hashes=hash_tree(BASE)
base_identity=hashlib.sha256(json.dumps(base_hashes,sort_keys=True,separators=(',',':')).encode()).hexdigest()
if sha(BASE/'PAD_DATA_AND_EVALUATION_CONTRACT_v2_CROSSDOMAIN_BINARY.md')!=V2_CONTRACT_SHA:raise RuntimeError('V2 contract SHA mismatch')
if sha(MODEL)!=DETECTOR_SHA:raise RuntimeError('Frozen SCRFD model SHA mismatch')
base_config=read_json(BASE/'config.json');base_prov=read_json(BASE/'provenance.json');base_fp=read_json(BASE/'manifest_fingerprints.json')
if base_prov.get('manifest_fingerprints')!=base_fp:raise RuntimeError('Base provenance manifest fingerprints mismatch')
if base_config.get('v2_contract_sha256')!=V2_CONTRACT_SHA or base_config.get('parent_contract_sha256')!=PARENT_CONTRACT_SHA:
    raise RuntimeError('Base config contract mismatch')
if base_prov.get('v2_contract_sha256')!=V2_CONTRACT_SHA or base_prov.get('parent_contract_sha256')!=PARENT_CONTRACT_SHA:
    raise RuntimeError('Base provenance contract mismatch')
if base_config.get('detector_sha256')!=DETECTOR_SHA or base_prov.get('detector_sha256')!=DETECTOR_SHA:
    raise RuntimeError('Base detector mismatch')
if base_prov.get('scrfd_execution_provider')!='CUDAExecutionProvider':raise RuntimeError('Base detector provider mismatch')
if base_config.get('input')!={'scale':2.7,'size':[80,80],'channels':'BGR','range':'[0,1]'}:
    raise RuntimeError('Base PAD input contract mismatch')
if sha(BASE/'bbox/celeba_bbox_cache_full.json')!=base_prov['celeba_bbox_sha256'] or sha(BASE/'bbox/lcc_official_evaluation_bbox_cache.json')!=base_prov['lcc_bbox_sha256']:
    raise RuntimeError('Base bbox cache SHA mismatch')
if sha(BASE/'labels/celeba_auxiliary_labels.csv')!=base_prov['auxiliary_labels_sha256']:
    raise RuntimeError('Base auxiliary label SHA mismatch')
for name,fp in base_fp.items():
    path=BASE/'manifests'/(name+'.csv')
    if not path.is_file() or sha(path)!=fp['file_sha256']:raise RuntimeError('Base manifest SHA mismatch: '+name)
    frame=pd.read_csv(path,keep_default_na=False)
    n=fp.get('sample_count',fp.get('candidate_n'))
    if len(frame)!=n:raise RuntimeError('Base manifest count mismatch: '+name)
    if name=='lcc_official_evaluation_full':
        if ordered_sha(frame.path)!=fp['ordered_path_sha256'] or int(frame.usable_for_pad.sum())!=fp['scored_n']:
            raise RuntimeError('Evaluation manifest fingerprint mismatch')
    else:
        if ordered_sha(frame.sample_key)!=fp['ordered_sample_key_sha256']:
            raise RuntimeError('CelebA manifest order mismatch: '+name)
        if 'class_counts' in fp and {str(k):int(v) for k,v in frame.three_class_label.value_counts().sort_index().items()}!=fp['class_counts']:
            raise RuntimeError('CelebA manifest class counts mismatch: '+name)
eval_frame=pd.read_csv(BASE/'manifests/lcc_official_evaluation_full.csv',keep_default_na=False)
if list(eval_frame.columns)!=['path','label','status','usable_for_pad','bbox_origin']:
    raise RuntimeError('Frozen LCC manifest schema mismatch')
if len(eval_frame)!=7580 or (eval_frame.label==0).sum()!=314 or (eval_frame.label==1).sum()!=7266:
    raise RuntimeError('Frozen LCC evaluation population mismatch')
eval_cache=read_json(BASE/'bbox/lcc_official_evaluation_bbox_cache.json')
if eval_cache.get('schema')!='lcc_official_evaluation_localization_v2' or set(eval_cache.get('records',{}))!=set(eval_frame.path):
    raise RuntimeError('Frozen LCC cache schema/keys mismatch')
eval_header=eval_cache['header'];policy=eval_header['policy']
if eval_header.get('provider')!='CUDAExecutionProvider' or eval_header.get('detector_sha256')!=DETECTOR_SHA or eval_header.get('contract_sha256')!=PARENT_CONTRACT_SHA:
    raise RuntimeError('Frozen LCC cache header mismatch')
if policy!={'schema':'external_scrfd_v1','detector_sha256':DETECTOR_SHA,'sizes':[[640,640],[480,480],[320,320]],
    'confidence':.5,'selection':'largest valid face; confidence then xyxy tie-break',
    'policy_version':'PAD_DATA_AND_EVALUATION_CONTRACT_v1','fallback':'none','provider':['CUDAExecutionProvider']}:
    raise RuntimeError('Frozen LCC detector policy mismatch')
if eval_header.get('source_hashes',{}).get('scrfd_model')!=DETECTOR_SHA:
    raise RuntimeError('Frozen LCC source detector mismatch')
for row in eval_frame.itertuples(index=False):
    rec=eval_cache['records'][row.path];check_fresh_record(rec,row.path)
    if (row.status,bool(row.usable_for_pad),row.bbox_origin)!=(rec['status'],rec['usable_for_pad'],rec['bbox_origin']):
        raise RuntimeError('Frozen LCC manifest/cache row mismatch: '+row.path)
if base_config['audit']['lcc_candidate_n']!=7580 or base_config['audit']['lcc_real_n']!=314 or base_config['audit']['lcc_attack_n']!=7266:
    raise RuntimeError('00c audit population mismatch')

if str(getattr(insightface,'__version__','unknown'))!='2.0':raise RuntimeError('Frozen InsightFace 2.0 required')
if 'CUDAExecutionProvider' not in ort.get_available_providers():
    raise RuntimeError('onnxruntime-gpu has no CUDAExecutionProvider. Check the installed ONNX Runtime build and CUDA/cuDNN compatibility; no CPU fallback.')
SCRFD_PRIMARY_INPUT_SIZE=(640,640);SCRFD_FALLBACK_INPUT_SIZES=[(480,480),(320,320)];SCRFD_CONF_THRESH=.5
EXPECTED_SCRFD_SHA256=DETECTOR_SHA;provider=['CUDAExecutionProvider'];ctx=0
scrfd_device=torch.cuda.get_device_name(0)
scrfd_session_providers=None # Filled only if this run initializes the detector.

# The following detector helpers are copied from 00-v2 without algorithm changes.
def _scrfd_detect(detector, image_bgr, input_size):
    """
    Prefer detect(..., input_size=...). Older InsightFace variants are handled
    by temporarily changing detector.input_size.
    """
    try:
        return detector.detect(
            image_bgr,
            input_size=tuple(input_size),
            max_num=0,
            metric="default",
        )
    except TypeError:
        old_input_size = getattr(detector, "input_size", None)
        detector.input_size = tuple(input_size)
        try:
            return detector.detect(image_bgr, max_num=0, metric="default")
        finally:
            detector.input_size = old_input_size


def prepare_scrfd():
    global scrfd_session_providers
    detector=model_zoo.get_model(str(Path(SCRFD_MODEL_PATH)),providers=provider)
    if detector is None:raise RuntimeError('Exact SCRFD-500M model unavailable')
    detector.prepare(ctx_id=ctx,input_size=SCRFD_PRIMARY_INPUT_SIZE,det_thresh=SCRFD_CONF_THRESH)
    session=getattr(detector,'session',None)
    active=session.get_providers() if session is not None else []
    # ORT 1.26 may register CPU for shape operators even when CUDA alone is requested.
    if not active or active[0]!='CUDAExecutionProvider' or set(active)-{'CUDAExecutionProvider','CPUExecutionProvider'}:
        raise RuntimeError('SCRFD requires CUDA as primary execution provider; active providers: '+str(active))
    if not hasattr(session,'disable_fallback'):
        raise RuntimeError('Cannot disable ONNX Runtime whole-session CPU fallback')
    session.disable_fallback()
    scrfd_session_providers=list(active)
    print('SCRFD primary provider:',active[0],'registered providers:',active,'device:',scrfd_device)
    return detector

def detect_lcc_record(detector,path):
    # Exact 00-v2 LCC size-order, face filter, tie-break and record fields.
    image=cv2.imread(str(LCC/path),cv2.IMREAD_COLOR)
    rec=sanitized_invalid('UNREADABLE',path,'image_read_failure') if image is None else {'status':'PENDING'}
    if image is not None:
        for size in ((640,640),(480,480),(320,320)):
            detections,_=_scrfd_detect(detector,image,size)
            valid=[]
            for det in (detections if detections is not None else []):
                x1,y1,x2,y2,confidence=map(float,det[:5])
                if all(np.isfinite((x1,y1,x2,y2,confidence))) and confidence>=.5 and x2>x1 and y2>y1:valid.append(((x2-x1)*(y2-y1),confidence,(x1,y1,x2,y2)))
            if valid:
                valid.sort(key=lambda v:(-v[0],-v[1],*v[2]));_,confidence,box=valid[0]
                rec={'status':'VALID','source_key':path,'bbox_origin':'SCRFD','bbox_source':'SCRFD','usable_for_pad':True,'bbox_xyxy':list(box),'confidence':confidence,'n_faces':len(valid),'ambiguous_multiface':len(valid)>1,'det_input_size':list(size)}
                break
        if rec['status']=='PENDING':rec=sanitized_invalid('DET_FAIL',path,'no_scrfd_detection')
    check_fresh_record(rec,path)
    return rec

def scan_split(split):
    rows=[];extensions={'.jpg','.jpeg','.png','.bmp','.webp'} # Exact 00-v2 LCC loader extensions.
    split_root=LCC/('LCC_FASD_'+split)
    if not split_root.is_dir():raise FileNotFoundError(split_root)
    for label,dirname in ((0,'real'),(1,'spoof')):
        folder=split_root/dirname
        if not folder.is_dir():raise FileNotFoundError(folder)
        for image_path in sorted(folder.rglob('*')):
            if image_path.is_file() and image_path.suffix.lower() in extensions:
                rows.append((image_path.relative_to(LCC).as_posix(),label))
    if len(rows)!=len({p for p,_ in rows}):raise RuntimeError('Duplicate LCC paths: '+split)
    expected={'training':(8299,1223,7076),'development':(2948,405,2543)}[split]
    if (len(rows),sum(y==0 for _,y in rows),sum(y==1 for _,y in rows))!=expected:
        raise RuntimeError(f'Unexpected raw LCC {split} population; expected {expected}')
    return rows

def candidate_fingerprint(rows):
    return {'candidate_n':len(rows),'real_n':sum(y==0 for _,y in rows),
        'spoof_n':sum(y==1 for _,y in rows),'ordered_relative_path_sha256':ordered_sha(p for p,_ in rows),
        'ordered_descriptor_sha256':descriptor_sha(rows)}

split_rows={s:scan_split(s) for s in ('training','development')}
eval_raw_root=LCC/'LCC_FASD_evaluation'
if not eval_raw_root.is_dir():raise FileNotFoundError(eval_raw_root)
eval_raw=[]
for label,dirname in ((0,'real'),(1,'spoof')):
    folder=eval_raw_root/dirname
    if not folder.is_dir():raise FileNotFoundError(folder)
    for image_path in sorted(folder.rglob('*')):
        if image_path.is_file() and image_path.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}:
            eval_raw.append((image_path.relative_to(eval_raw_root).as_posix(),label))
if len(eval_raw)!=7580 or sum(y==0 for _,y in eval_raw)!=314 or sum(y==1 for _,y in eval_raw)!=7266:
    raise RuntimeError('Mounted LCC evaluation raw population mismatch')
if [p for p,_ in eval_raw]!=eval_frame.path.astype(str).tolist() or [y for _,y in eval_raw]!=eval_frame.label.astype(int).tolist():
    raise RuntimeError('Mounted LCC evaluation differs from frozen 00c manifest')
candidate_fps={s:candidate_fingerprint(rows) for s,rows in split_rows.items()}
candidate_fps['evaluation']=candidate_fingerprint(eval_raw)
if candidate_fps['evaluation']['ordered_relative_path_sha256']!=base_fp['lcc_official_evaluation_full']['ordered_path_sha256']:
    raise RuntimeError('Frozen evaluation candidate fingerprint mismatch')
if sum(x['candidate_n'] for x in candidate_fps.values())!=18827 or sum(x['real_n'] for x in candidate_fps.values())!=1942:
    raise RuntimeError('Combined official population mismatch')

# Copy all inherited resources without editing any existing base-relative file.
if ROOT.exists() and not ROOT.is_dir():raise RuntimeError('OUTPUT_ROOT exists but is not a directory')
if not ROOT.exists():shutil.copytree(BASE,ROOT,copy_function=shutil.copy2)
else:
    for rel in base_hashes:
        src=BASE/rel;dst=ROOT/rel
        if not dst.exists():dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,dst)
        if not dst.is_file() or sha(dst)!=base_hashes[rel]:raise RuntimeError('Inherited resource drift: '+rel)
def verify_inherited():
    after={rel:sha(ROOT/rel) for rel in base_hashes}
    if after!=base_hashes:raise RuntimeError('Original 00c resource bytes changed')
    if after['bbox/lcc_official_evaluation_bbox_cache.json']!=base_prov['lcc_bbox_sha256'] or after['manifests/lcc_official_evaluation_full.csv']!=base_fp['lcc_official_evaluation_full']['file_sha256']:
        raise RuntimeError('Evaluation cache/manifest changed')
    return after
verify_inherited()
atomic_json_write(ROOT/'base_resource_hashes_before.json',base_hashes)
EXTENSION_PROTOCOL_TEXT = "# LCC pooled resource extension protocol v1\n\nStatus: frozen additive resource extension, 2026-10-03. This extends the completed `minifasnet_binary_crossdomain_resources_v2` bundle. It does not replace or edit `PAD_DATA_AND_EVALUATION_CONTRACT_v2_CROSSDOMAIN_BINARY.md` (SHA256 `2f1f6eb6351550d491e4fb59e2151b1ccd9b000c0985cd32f12621f9d58c5f91`) or its parent v1 contract (SHA256 `aa67032a79627636d967dc885202540a34410c2b4d64522a5604d3aedff85b2d`). All pre-existing files, including CelebA resources and the official LCC evaluation bbox cache and manifest, are copied byte for byte and verified by recursive SHA256 before and after extension. Original `config.json`, `provenance.json`, and `manifest_fingerprints.json` remain byte identical; extension metadata is additive.\n\n## Purpose and data roles\n\nThe extension enables post-hoc split-wise and pooled cross-domain PAD evaluation on `LCC_FASD_training`, `LCC_FASD_development`, and the frozen `LCC_FASD_evaluation`. The pooled set is a legacy-repo-aligned diagnostic target population, not a replacement for the official LCC evaluation split. These LCC splits must never train or fine-tune CelebA models, choose checkpoints, or calibrate thresholds. Future PAD evaluation must retain the model's locked CelebA-Val threshold. This notebook performs localization and cache construction only; it does not load PAD checkpoints, run PAD inference, calculate PAD metrics, or export image crops.\n\nExpected raw populations are training 1,223 Real + 7,076 Spoof = 8,299; development 405 Real + 2,543 Spoof = 2,948; and unchanged evaluation 314 Real + 7,266 Spoof = 7,580. Combined: 1,942 Real + 16,885 Spoof = 18,827. Assert these counts before new localization and retain every candidate, including failures. Training/development labels come only from `real/` and `spoof/` directories. Training and development keys are lexically ordered POSIX paths relative to the common LCC root, with split prefixes. Evaluation keys remain the original `real/...` and `spoof/...` paths relative to `LCC_FASD_evaluation`; `lcc_split` in the combined manifest specifies how future evaluators resolve each path. Evaluation records and manifest rows are copied, never re-localized or rewritten.\n\n## Frozen detector and record contract\n\nOnly training and development receive new SCRFD inference. Reuse the exact LCC localization functions and selection loop from `00-v2_prepare_minifasnet_full_resources.ipynb` (source SHA256 `e4c8565f48be3ee82228b5e88666c3cb0001ce92295835e973253fc3087b495d`): SCRFD-500M model SHA256 `5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a`, InsightFace 2.0, CUDAExecutionProvider requested and required as the primary active provider, confidence 0.5, input sizes 640 → 480 → 320, select the largest finite positive-area face at the first successful size, then confidence and xyxy as tie breakers. The original `_scrfd_detect` API compatibility path and LCC candidate selection/record fields are retained. ONNX Runtime 1.26 may register CPUExecutionProvider secondarily for shape operators even when only CUDA is requested. The 00d provider check accepts that registration, rejects a CPU-only session, and disables ONNX Runtime's whole-session runtime fallback. The registered providers are logged in extension provenance. There is no CPU-only detector run, GT, alternate detector, center-crop, or full-image fallback.\n\nScorable records require `status=VALID`, `usable_for_pad=True`, `bbox_origin=SCRFD`, `bbox_source` beginning `SCRFD`, matching `source_key`, finite positive-area raw `bbox_xyxy`, and no `crop_factor`. Failures retain the original semantics: `DET_FAIL`, `SUSPICIOUS`, `UNREADABLE`, `INVALID_BBOX`; no failure becomes a valid crop. The new split caches preserve the official cache's `schema/header/records` shape and frozen policy/header values, with additive source-split, candidate-fingerprint, and extension-protocol fields. The combined cache is the exact record union of new training/development caches and the untouched evaluation cache; its evaluation record values must compare equal to the original values. No 2.7× crop or 80×80 image is cached.\n\n## Copying, resume, and audits\n\nBefore detector inference, verify the complete base 00c bundle: contracts, detector/provider/threshold/sizes/input policy, bbox-cache hashes, every original manifest fingerprint, and official evaluation population. Hash all base files, copy the bundle to a distinct output directory, and verify copied file hashes. Repeat the inherited-file hash audit after extension. The original evaluation cache and manifest SHA256 must match before and after. Fail on any mismatch.\n\nFor each new split, fingerprint the lexically ordered relative paths and ordered `(path,label)` descriptors before inference. `SPLIT_FILTER=None` processes both; `['training']` and `['development']` process one. A partial cache has the same header and record schema as the completed cache and is written atomically after at most 200 new records. Resume verifies the detector, policy, split, candidate fingerprints and existing records, then localizes only missing keys, whether prior records are valid or failed. A completed split is reused after the same checks. Each completed split gets its manifest immediately. The combined cache, pooled manifest, final extension metadata, and ZIP are emitted only when both new splits are complete and every invariant passes.\n\nWrite `lcc_training_full.csv`, `lcc_development_full.csv`, and `lcc_combined_full.csv` with the official evaluation manifest's original columns intact. The combined manifest appends `lcc_split` and orders training, development, then evaluation. Every manifest path must map to exactly one cache record with matching status, usability, and bbox origin. Report candidate/scored counts, class-wise detector coverage, and all five localization status counts for each split and the pool. Save additive extension config, provenance, fingerprints, and before/after base-file hashes. The final ZIP is created only after successful validation.\n"
if hashlib.sha256(EXTENSION_PROTOCOL_TEXT.encode()).hexdigest()!=EXTENSION_PROTOCOL_SHA:
    raise RuntimeError('Embedded extension protocol SHA mismatch')
protocol_path=ROOT/'LCC_POOLED_RESOURCE_EXTENSION_PROTOCOL_v1.md'
PREVIOUS_PROTOCOL_SHA='34a792d1ce13164aee20ec3a984f56e252eabab63a4309911635249e770db7ac'
new_split_cache_files=[ROOT/'bbox'/f'lcc_{split}_bbox_cache{suffix}.json'
                       for split in ('training','development') for suffix in ('','.partial')]
can_upgrade_failed_preflight=not any(path.exists() for path in new_split_cache_files)
if protocol_path.exists() and sha(protocol_path)!=EXTENSION_PROTOCOL_SHA:
    if sha(protocol_path)!=PREVIOUS_PROTOCOL_SHA or not can_upgrade_failed_preflight:
        raise RuntimeError('Extension protocol drift; existing detection cache requires its original protocol')
protocol_path.write_text(EXTENSION_PROTOCOL_TEXT)

extension_config={'schema':'lcc_pooled_resource_extension_v1','base_resource_dir':str(BASE),
    'base_resource_identity_sha256':base_identity,
    'base_config_sha256':base_hashes['config.json'],'base_provenance_sha256':base_hashes['provenance.json'],
    'base_manifest_fingerprints_sha256':base_hashes['manifest_fingerprints.json'],
    'v2_contract_sha256':V2_CONTRACT_SHA,'parent_contract_sha256':PARENT_CONTRACT_SHA,
    'extension_protocol_sha256':EXTENSION_PROTOCOL_SHA,'localization_notebook_sha256':LOCALIZATION_NOTEBOOK_SHA,
    'detector_sha256':DETECTOR_SHA,'provider':['CUDAExecutionProvider'],'threshold':.5,
    'sizes':[[640,640],[480,480],[320,320]],'lcc_root':str(LCC),'candidate_fingerprints':candidate_fps,
    'save_every':CACHE_SAVE_EVERY}
config_path=ROOT/'lcc_pooled_extension_config.json'
if config_path.exists() and read_json(config_path)!=extension_config:
    previous_config={**extension_config,'extension_protocol_sha256':PREVIOUS_PROTOCOL_SHA}
    if read_json(config_path)!=previous_config or not can_upgrade_failed_preflight:
        raise RuntimeError('Extension config drift')
atomic_json_write(config_path,extension_config)

def split_header(split):
    fp=candidate_fps[split]
    header={k:v for k,v in eval_header.items() if k!='source_hashes'}
    header['source_hashes']={k:v for k,v in eval_header['source_hashes'].items() if k!='official_lcc_paths'}
    header['source_hashes']['lcc_'+split+'_paths']=fp['ordered_relative_path_sha256']
    header.update({'source_split':split,'cache_schema':'lcc_scrfd_localization_extension_v1',
                   'candidate_fingerprint':fp,'extension_protocol_sha256':EXTENSION_PROTOCOL_SHA})
    if header['policy']!=eval_header['policy'] or header['provider']!=eval_header['provider']:
        raise RuntimeError('Split header detector drift')
    return header

def read_split_payload(path,header,expected_keys,complete):
    payload=read_json(path)
    if payload.get('schema')!='lcc_scrfd_localization_extension_v1' or payload.get('header')!=header:
        raise RuntimeError('Split cache header/schema mismatch: '+str(path))
    records=payload.get('records')
    if not isinstance(records,dict) or not set(records)<=expected_keys or (complete and set(records)!=expected_keys):
        raise RuntimeError('Split cache candidate mismatch: '+str(path))
    for key,rec in records.items():check_fresh_record(rec,key)
    return records

def process_split(split):
    rows=split_rows[split];expected=set(p for p,_ in rows);header=split_header(split)
    finished=ROOT/'bbox'/('lcc_'+split+'_bbox_cache.json')
    partial=ROOT/'bbox'/('lcc_'+split+'_bbox_cache.partial.json')
    if finished.exists():
        records=read_split_payload(finished,header,expected,True)
        if partial.exists():
            recovered=read_split_payload(partial,header,expected,False)
            if any(records[k]!=rec for k,rec in recovered.items()):raise RuntimeError('Finished/partial cache conflict')
            partial.unlink()
        print(split,'cache already complete; detector inference skipped')
        return records
    records=read_split_payload(partial,header,expected,False) if partial.exists() else {}
    missing=[p for p,_ in rows if p not in records]
    detector=prepare_scrfd() if missing else None
    try:
        for index,path in enumerate(missing,1):
            # Detector exceptions are not converted to failures; retry the key after investigation.
            rec=detect_lcc_record(detector,path)
            records[path]=rec
            if index%CACHE_SAVE_EVERY==0:
                atomic_json_write(partial,{'schema':'lcc_scrfd_localization_extension_v1','header':header,
                                           'records':{k:records[k] for k in sorted(records)}})
                print(split,'localized',len(records),'/',len(rows),flush=True)
        if set(records)!=expected:raise RuntimeError('Incomplete split cache: '+split)
        atomic_json_write(finished,{'schema':'lcc_scrfd_localization_extension_v1','header':header,
                                   'records':{k:records[k] for k in sorted(records)}})
        if partial.exists():partial.unlink()
        return records
    finally:
        if detector is not None:del detector

for split in (('training','development') if SPLIT_FILTER is None else SPLIT_FILTER):
    process_split(split)

def manifest_of(rows,records):
    frame=pd.DataFrame(rows,columns=['path','label'])
    frame['status']=[records[p]['status'] for p in frame.path]
    frame['usable_for_pad']=[bool(records[p]['usable_for_pad']) for p in frame.path]
    frame['bbox_origin']=[records[p]['bbox_origin'] for p in frame.path]
    if list(frame.columns)!=list(eval_frame.columns):raise RuntimeError('LCC manifest schema drift')
    return frame

def verify_manifest_cache(frame,records,name):
    if len(frame)!=len(records) or set(frame.path)!=set(records):raise RuntimeError('Manifest/cache key mismatch: '+name)
    for row in frame.itertuples(index=False):
        rec=records[row.path];check_fresh_record(rec,row.path)
        if (row.status,bool(row.usable_for_pad),row.bbox_origin)!=(rec['status'],rec['usable_for_pad'],rec['bbox_origin']):
            raise RuntimeError('Manifest/cache state mismatch: '+row.path)

def coverage(frame):
    result={'candidate_n':len(frame),'scored_n':int(frame.usable_for_pad.sum()),
            'detector_coverage':float(frame.usable_for_pad.mean()),
            'status_counts':{status:int((frame.status==status).sum()) for status in ('VALID','DET_FAIL','SUSPICIOUS','UNREADABLE','INVALID_BBOX')}}
    for label,key in ((0,'real'),(1,'spoof')):
        part=frame.loc[frame.label==label]
        result[key]={'candidate_n':len(part),'scored_n':int(part.usable_for_pad.sum()),
                     'detector_coverage':float(part.usable_for_pad.mean())}
    return result

finished_paths={s:ROOT/'bbox'/('lcc_'+s+'_bbox_cache.json') for s in ('training','development')}
for s,path in finished_paths.items():
    if path.is_file():
        records=read_split_payload(path,split_header(s),set(p for p,_ in split_rows[s]),True)
        frame=manifest_of(split_rows[s],records)
        verify_manifest_cache(frame,records,s)
        manifest_path=ROOT/'manifests'/('lcc_'+s+'_full.csv')
        if manifest_path.exists():
            if not pd.read_csv(manifest_path,keep_default_na=False).equals(frame):
                raise RuntimeError('Existing new split manifest drift: '+s)
        else:frame.to_csv(manifest_path,index=False)
if not all(path.is_file() for path in finished_paths.values()):
    verify_inherited()
    print('Selected split(s) processed. Both complete split caches are required for combined manifests and final ZIP.')
else:
    split_records={s:read_split_payload(finished_paths[s],split_header(s),set(p for p,_ in split_rows[s]),True)
                   for s in ('training','development')}
    frames={s:manifest_of(split_rows[s],split_records[s]) for s in ('training','development')}
    frames['evaluation']=eval_frame.copy()
    for s in frames:verify_manifest_cache(frames[s],split_records[s] if s!='evaluation' else eval_cache['records'],s)
    combined_records={}
    for records in (split_records['training'],split_records['development'],eval_cache['records']):
        if set(combined_records)&set(records):raise RuntimeError('Duplicate pooled LCC cache key')
        combined_records.update(records)
    if len(combined_records)!=18827:raise RuntimeError('Pooled LCC cache count mismatch')
    for key,rec in eval_cache['records'].items():
        if combined_records[key]!=rec:raise RuntimeError('Evaluation record changed in pool: '+key)
    combined=pd.concat([frames[s].assign(lcc_split=s) for s in ('training','development','evaluation')],ignore_index=True)
    if len(combined)!=18827 or int((combined.label==0).sum())!=1942 or int((combined.label==1).sum())!=16885:
        raise RuntimeError('Pooled LCC manifest population mismatch')
    verify_manifest_cache(combined,combined_records,'combined')
    combined_path=ROOT/'manifests/lcc_combined_full.csv'
    if combined_path.exists():
        if not pd.read_csv(combined_path,keep_default_na=False).equals(combined):raise RuntimeError('Existing pooled manifest drift')
    else:combined.to_csv(combined_path,index=False)
    combined_cache_path=ROOT/'bbox/lcc_combined_bbox_cache.json'
    combined_header={k:v for k,v in eval_header.items() if k!='source_hashes'}
    combined_header.update({'source_split':'combined','cache_schema':'lcc_combined_localization_v1',
        'extension_protocol_sha256':EXTENSION_PROTOCOL_SHA,
        'component_cache_sha256':{'training':sha(finished_paths['training']),
                                  'development':sha(finished_paths['development']),
                                  'evaluation':base_prov['lcc_bbox_sha256']}})
    if combined_cache_path.exists():
        payload=read_json(combined_cache_path)
        if payload.get('schema')!='lcc_combined_localization_v1' or payload.get('header')!=combined_header or payload.get('records')!=combined_records:
            raise RuntimeError('Existing pooled cache drift')
    else:
        atomic_json_write(combined_cache_path,{'schema':'lcc_combined_localization_v1','header':combined_header,
                                               'records':{k:combined_records[k] for k in sorted(combined_records)}})
    audits={s:coverage(frames[s]) for s in frames}
    audits['combined']=coverage(combined)
    atomic_json_write(ROOT/'lcc_pooled_extension_coverage_audit.json',audits)
    def manifest_fp(name,frame,path):
        audit=coverage(frame)
        return {'file_sha256':sha(path),'candidate_n':audit['candidate_n'],'scored_n':audit['scored_n'],
                'detector_coverage':audit['detector_coverage'],'real_n':audit['real']['candidate_n'],
                'spoof_n':audit['spoof']['candidate_n'],'ordered_path_sha256':ordered_sha(frame.path)}
    def cache_fp(path,records,header,candidate_fp):
        return {'file_sha256':sha(path),'record_count':len(records),
                'valid_count':sum(r['status']=='VALID' for r in records.values()),
                'detector_sha256':header['detector_sha256'],'provider':header['provider'],
                'candidate_fingerprint':candidate_fp}
    manifest_paths={s:ROOT/'manifests'/('lcc_'+s+'_full.csv') for s in ('training','development')}
    manifest_paths['evaluation']=ROOT/'manifests/lcc_official_evaluation_full.csv'
    manifest_paths['combined']=combined_path
    cache_paths={**finished_paths,'evaluation':ROOT/'bbox/lcc_official_evaluation_bbox_cache.json',
                 'combined':combined_cache_path}
    cache_records={**split_records,'evaluation':eval_cache['records'],'combined':combined_records}
    cache_headers={s:split_header(s) for s in ('training','development')}
    cache_headers['evaluation']=eval_header;cache_headers['combined']=combined_header
    pooled_candidates=[*split_rows['training'],*split_rows['development'],*eval_raw]
    pooled_fp=candidate_fingerprint(pooled_candidates)
    manifest_names={s:('lcc_official_evaluation_full' if s=='evaluation' else 'lcc_'+s+'_full')
                    for s in ('training','development','evaluation','combined')}
    cache_names={s:('lcc_official_evaluation_bbox_cache' if s=='evaluation' else 'lcc_'+s+'_bbox_cache')
                 for s in ('training','development','evaluation','combined')}
    extension_fps={**{manifest_names[s]:manifest_fp(s,frames[s] if s!='combined' else combined,manifest_paths[s])
                      for s in manifest_names},
                   **{cache_names[s]:cache_fp(cache_paths[s],cache_records[s],cache_headers[s],
                                              pooled_fp if s=='combined' else candidate_fps[s])
                      for s in cache_names}}
    atomic_json_write(ROOT/'lcc_pooled_extension_fingerprints.json',extension_fps)
    before_eval_cache=base_hashes['bbox/lcc_official_evaluation_bbox_cache.json']
    before_eval_manifest=base_hashes['manifests/lcc_official_evaluation_full.csv']
    after_hashes=verify_inherited()
    if after_hashes['bbox/lcc_official_evaluation_bbox_cache.json']!=before_eval_cache or after_hashes['manifests/lcc_official_evaluation_full.csv']!=before_eval_manifest:
        raise RuntimeError('Evaluation resource was regenerated')
    atomic_json_write(ROOT/'base_resource_hashes_after.json',after_hashes)
    extension_prov={'schema':'lcc_pooled_resource_extension_v1','base_resource_dir':str(BASE),
        'base_resource_identity_sha256':base_identity,
        'base_v2_contract_sha256':V2_CONTRACT_SHA,
        'extension_protocol_sha256':EXTENSION_PROTOCOL_SHA,'detector_sha256':DETECTOR_SHA,
        'detector_provider':'CUDAExecutionProvider','detector_threshold':.5,
        'detector_sizes':[[640,640],[480,480],[320,320]],'lcc_root':str(LCC),
        'source_candidate_fingerprints':candidate_fps,
        'split_cache_sha256':{s:sha(cache_paths[s]) for s in cache_paths},
        'split_manifest_sha256':{s:sha(manifest_paths[s]) for s in manifest_paths},
        'combined_cache_sha256':sha(combined_cache_path),
        'combined_manifest_sha256':sha(combined_path),
        'base_config_sha256':base_hashes['config.json'],'base_provenance_sha256':base_hashes['provenance.json'],
        'base_manifest_fingerprints_sha256':base_hashes['manifest_fingerprints.json'],
        'localization_notebook_sha256':LOCALIZATION_NOTEBOOK_SHA,
        'versions':{'python':sys.version.split()[0],'torch':torch.__version__,'cuda':torch.version.cuda,
                    'insightface':insightface.__version__,'onnxruntime':ort.__version__,
                    'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__},
        'cuda_available':bool(torch.cuda.is_available()),'cuda_device':scrfd_device,
        'scrfd_session_providers_this_run':scrfd_session_providers,
        'onnxruntime_available_providers':ort.get_available_providers()}
    atomic_json_write(ROOT/'lcc_pooled_extension_provenance.json',extension_prov)
    verify_inherited()
    zip_path=ROOT.parent/(ROOT.name+'.zip')
    tmp_zip=zip_path.with_suffix('.zip.tmp')
    if tmp_zip.exists():tmp_zip.unlink()
    try:
        with zipfile.ZipFile(tmp_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as archive:
            for path in sorted(ROOT.rglob('*')):
                if path.is_file():archive.write(path,arcname=(Path(ROOT.name)/path.relative_to(ROOT)).as_posix())
        os.replace(tmp_zip,zip_path)
    finally:
        if tmp_zip.exists():tmp_zip.unlink()
    print('split         candidates  scored  coverage  real  spoof')
    for s in ('training','development','evaluation','combined'):
        a=audits[s]
        print(f"{s:12s} {a['candidate_n']:10d} {a['scored_n']:7d} {a['detector_coverage']:.6f} {a['real']['candidate_n']:5d} {a['spoof']['candidate_n']:6d}")
        for label in ('real','spoof'):
            cls=a[label]
            print(f"  {label}: candidates={cls['candidate_n']} scored={cls['scored_n']} coverage={cls['detector_coverage']:.6f}")
        print('  statuses:',a['status_counts'])
    for s in cache_paths:print(s,'cache SHA256',sha(cache_paths[s]),'manifest SHA256',sha(manifest_paths[s]))
    print('extension protocol SHA256',EXTENSION_PROTOCOL_SHA)
    print('EXISTING LCC EVALUATION CACHE REUSED: YES')
    print('EXISTING 00c BASE FILES MODIFIED: NO')
    display(FileLink(str(zip_path)))
